01 · Construção da base

Transforma os microdados brutos da POF 2017-2018 (IBGE) em uma tabela com uma linha por ocasião de consumo, com o target e as features prontas para o modelo.

Por que o target é "pizza" e não "pizza + guaraná"?

Na POF há só 184 ocasiões com pizza + guaraná em ~363 mil ocasiões o que é raro demais para um modelo aprender com segurança.
Com 1.711 ocasiões de pizza, o modelo aprende onde a ocasião acontece. O Guaraná entra em duas métricas por segmento que alimentam o Decision Engine:
- attach de refrigerante: % das ocasiões de pizza que têm refrigerante;
- share de Guaraná: % das ocasiões pizza + refri em que o refri é de guaraná.

1. Imports e leitura

In [ ]:
# importa bibliotecas
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pof import ler_registro, cadastro_produtos

pd.set_option("display.max_columns", 50) # mostra até 50 colunas

# pasta onde serão salvos
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# identifica uma pessoa na POF
CHAVE_PESSOA = ["COD_UPA", "NUM_DOM", "NUM_UC", "COD_INFORMANTE"]

# colunas usadas
cols = ["UF", "ESTRATO_POF", *CHAVE_PESSOA, "QUADRO", "V9001", "V9015", "V9017", "V9018",
        "RENDA_TOTAL", "DIA_SEMANA", "PESO_FINAL"]

# cada linha indica um item consumido por pessoa
consumo = ler_registro("Consumo Alimentar", cols)
print(f"{len(consumo):,} itens consumidos registrados")
consumo.head()

1,175,390 itens consumidos registrados


,UF,ESTRATO_POF,COD_UPA,NUM_DOM,NUM_UC,COD_INFORMANTE,QUADRO,V9001,V9015,V9017,V9018,PESO_FINAL,RENDA_TOTAL,DIA_SEMANA
0,11,1101,110000016,2,1,1,72,8000101,8H,1,1,2390.48652303,3855.34,Quarta-Feira
1,11,1101,110000016,2,1,1,72,7901602,8H,1,1,2390.48652303,3855.34,Quarta-Feira
2,11,1101,110000016,2,1,1,72,8501303,8H,1,1,2390.48652303,3855.34,Quarta-Feira
3,11,1101,110000016,2,1,1,72,6906602,8H,1,1,2390.48652303,3855.34,Quarta-Feira
4,11,1101,110000016,2,1,1,72,6300101,12H,2,1,2390.48652303,3855.34,Quarta-Feira


Cada linha é **um alimento** que uma pessoa consumiu. Variáveis principais:

| Variável | Significado |
|---|---|
| `QUADRO` | 72 = 1º dia do registro, 73 = 2º dia |
| `V9001` | código do alimento |
| `V9015` | horário (ex.: `20H`) |
| `V9017` | ocasião: 1 café, 2 almoço, 3 lanche, 4 jantar, 5 ceia, 6 outra, 9 não determinado |
| `V9018` | local: 1 em casa, 2 merenda, 3 rest. quilo, 4 lanchonete/fast food, 5 restaurante, 6 ambulante, 7 fora de casa outros |
| `PESO_FINAL` | peso amostral (quantas pessoas da população cada entrevistado representa) |

2. Identificar pizza, refrigerante e guaraná

In [ ]:
# usa cadastro dos produtos, e traduz o código do alimento p/ descrição
produtos = cadastro_produtos()
desc = dict(zip(produtos["codigo"], produtos["descricao"]))
consumo["descricao"] = consumo["V9001"].map(desc).fillna("")

d = consumo["descricao"]
consumo["is_pizza"] = d.str.contains("PIZZA") #identifica pizza

# identifica refri
consumo["is_refri"] = (
    d.str.contains(r"REFRIGERANTE|^GUARANA (?:TRADICIONAL|LIGHT|DIET)$|^SPRITE", regex=True)
    & ~d.str.contains("REFRESCO")
)

# identifica guarana
consumo["is_guarana"] = consumo["is_refri"] & d.str.contains("GUARANA")

# as 10 descrições (alimentos) de refri com + frequencia
print("Refrigerantes encontrados:")
print(consumo.loc[consumo.is_refri, "descricao"].value_counts().head(10))

Refrigerantes encontrados:
descricao
REFRIGERANTE DE COLA TRADICIONAL       5465
REFRIGERANTE DE GUARANA TRADICIONAL    4330
REFRIGERANTE NAO ESPECIFICADO          1159
REFRIGERANTE DE LARANJA TRADICIONAL     918
REFRIGERANTE DE UVA TRADICIONAL         237
REFRIGERANTE DE LIMAO TRADICIONAL       237
GUARANA TRADICIONAL                     206
REFRIGERANTE DE COLA DIET                88
SPRITE REFRIGERANTE TRADICIONAL          67
REFRIGERANTE DE GUARANA DIET             36
Name: count, dtype: int64


3. Agregar: de item para ocasião

Agrupa pela chave pessoa + dia + refeição. Para cada ocasião:
- contexto (dia da semana, horário, local) → vira feature;
- o que foi consumido (pizza, refri, guaraná) → vira target 

In [ ]:
consumo["hora"] = pd.to_numeric(consumo["V9015"].str.replace("H", ""), errors="coerce")
# ocasião é pessoa + dia + tipo ref
CHAVE_OCASIAO = CHAVE_PESSOA + ["QUADRO", "V9017"]


# muda como apresneta agora, item -> ocasião
ocasioes = (
    consumo.groupby(CHAVE_OCASIAO)
    .agg(
        uf=("UF", "first"),
        estrato=("ESTRATO_POF", "first"),
        dia_semana=("DIA_SEMANA", "first"),
        hora=("hora", "median"),
        local=("V9018", lambda s: s.mode().iat[0]),
        renda_total_uc=("RENDA_TOTAL", "first"),
        peso=("PESO_FINAL", "first"),
        n_itens=("V9001", "size"),
        y_pizza=("is_pizza", "max"),
        tem_refri=("is_refri", "max"),
        tem_guarana=("is_guarana", "max"),
    )
    .reset_index()
    .rename(columns={"V9017": "refeicao"})
)
print(f"{len(ocasioes):,} ocasiões")
ocasioes.head()

363,412 ocasiões


,COD_UPA,NUM_DOM,NUM_UC,COD_INFORMANTE,QUADRO,refeicao,uf,estrato,dia_semana,hora,local,renda_total_uc,peso,n_itens,y_pizza,tem_refri,tem_guarana
0,110000016,12,1,2,72,1,11,1101,Segunda-Feira,12.0,1,4353.51,1292.88962827,3,False,False,False
1,110000016,12,1,2,72,2,11,1101,Segunda-Feira,15.0,1,4353.51,1292.88962827,5,False,False,False
2,110000016,12,1,2,72,4,11,1101,Segunda-Feira,20.0,1,4353.51,1292.88962827,1,False,False,False
3,110000016,12,1,2,72,9,11,1101,Segunda-Feira,17.5,1,4353.51,1292.88962827,4,False,False,False
4,110000016,12,1,2,73,1,11,1101,Domingo,8.0,1,4353.51,1292.88962827,2,False,False,False


4. Features da pessoa e da família (registro "Morador")

In [6]:
mor = ler_registro("Morador", ["TIPO_SITUACAO_REG", *CHAVE_PESSOA, "V0403", "V0404",
                               "ANOS_ESTUDO", "RENDA_DISP_PC"])
mor["idade"] = pd.to_numeric(mor["V0403"], errors="coerce")
mor["anos_estudo"] = pd.to_numeric(mor["ANOS_ESTUDO"], errors="coerce")
mor["renda_pc"] = pd.to_numeric(mor["RENDA_DISP_PC"], errors="coerce")
mor["sexo"] = mor["V0404"].map({"1": "M", "2": "F"})
mor["urbano"] = (mor["TIPO_SITUACAO_REG"] == "1").astype(int)

# composição da unidade de consumo (família)
uc = mor.groupby(["COD_UPA", "NUM_DOM", "NUM_UC"]).agg(
    n_moradores=("idade", "size"),
    n_criancas=("idade", lambda s: (s < 12).sum()),
    n_adolescentes=("idade", lambda s: ((s >= 12) & (s < 18)).sum()),
).reset_index()

mor = mor.merge(uc, on=["COD_UPA", "NUM_DOM", "NUM_UC"], how="left")
mor = mor[[*CHAVE_PESSOA, "idade", "sexo", "anos_estudo", "renda_pc", "urbano",
           "n_moradores", "n_criancas", "n_adolescentes"]]

base = ocasioes.merge(mor, on=CHAVE_PESSOA, how="left", validate="many_to_one")
print("Linhas sem morador correspondente:", base["idade"].isna().sum())

Linhas sem morador correspondente: 0


5. Tratamento das variáveis e features derivadas

Features com significado de negócio:

| Feature | Hipótese |
|---|---|
| `fim_de_semana` | Pizza concentra-se de sexta a domingo (TAPI) |
| `periodo` | Noite é o momento da pizza |
| `em_casa` | A ocasião do TAPI é "dentro de casa" (inclui delivery) |
| `classe_renda` | TAPI: mais pizza na classe A → espaço em outras classes |
| `tem_crianca` | Pizza como refeição em família |
| `regiao` | Diferenças regionais de hábito |

In [7]:
DIAS = {"Domingo": 0, "Segunda-Feira": 1, "Terca-Feira": 2, "Quarta-Feira": 3,
        "Quinta-Feira": 4, "Sexta-Feira": 5, "Sabado": 6}
base["dia_semana_num"] = base["dia_semana"].map(DIAS)
base["fim_de_semana"] = base["dia_semana_num"].isin([0, 5, 6]).astype(int)  # sex, sáb, dom

def periodo(h):
    if pd.isna(h):
        return "indefinido"
    if 5 <= h < 11:
        return "manha"
    if 11 <= h < 15:
        return "almoco"
    if 15 <= h < 18:
        return "tarde"
    if 18 <= h < 23:
        return "noite"
    return "madrugada"

base["periodo"] = base["hora"].map(periodo)
base["refeicao"] = base["refeicao"].map({"1": "cafe", "2": "almoco", "3": "lanche", "4": "jantar",
                                         "5": "ceia", "6": "outra", "9": "indefinido"})
base["local"] = base["local"].map({"1": "casa", "2": "merenda", "3": "rest_quilo", "4": "lanchonete",
                                   "5": "restaurante", "6": "ambulante", "7": "fora_outros",
                                   "9": "indefinido"})
base["em_casa"] = (base["local"] == "casa").astype(int)

# Regiões pelo 1º dígito do código da UF (padrão IBGE)
base["regiao"] = base["uf"].str[0].map({"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul",
                                        "5": "Centro-Oeste"})
for c in ["renda_total_uc", "peso"]:
    base[c] = pd.to_numeric(base[c], errors="coerce")
base["tem_crianca"] = (base["n_criancas"] > 0).astype(int)

# Faixas de renda per capita (aproximação de "classe" por quantis ponderados não é necessária aqui)
base["classe_renda"] = pd.qcut(base["renda_pc"], 5, labels=["E", "D", "C", "B", "A"])

for c in ["y_pizza", "tem_refri", "tem_guarana"]:
    base[c] = base[c].astype(int)
base["pizza_refri"] = base["y_pizza"] * base["tem_refri"]
base["pizza_guarana"] = base["y_pizza"] * base["tem_guarana"]

# id de domicílio: usado depois na validação para a mesma família não cair em treino e teste
base["id_domicilio"] = base["COD_UPA"] + "_" + base["NUM_DOM"]
base["id_pessoa"] = base[CHAVE_PESSOA].agg("_".join, axis=1)

> Sobre `classe_renda`: são quintis da renda per capita (20% em cada faixa), rotulados de E a A. É uma aproximação, não o Critério Brasil

6. Checagens de qualidade

In [8]:
print("Formato:", base.shape)
print("\nTaxa de pizza por ocasião: {:.3%}".format(base["y_pizza"].mean()))
print("Ocasiões com pizza:", base["y_pizza"].sum())
print("Pizza + refri:", base["pizza_refri"].sum(), "| Pizza + guaraná:", base["pizza_guarana"].sum())
print("\nNulos por coluna (só as que têm):")
print(base.isna().sum()[lambda s: s > 0])

Formato: (363412, 36)

Taxa de pizza por ocasião: 0.471%
Ocasiões com pizza: 1711
Pizza + refri: 633 | Pizza + guaraná: 184

Nulos por coluna (só as que têm):
renda_pc        118
classe_renda    118
dtype: int64


7. Primeiros números de negócio

O peso amostral para estimar a população.

In [9]:
def taxa_ponderada(df, col, peso="peso"):
    return np.average(df[col], weights=df[peso])

pz = base[base["y_pizza"] == 1]
attach = taxa_ponderada(pz, "tem_refri")
pr = pz[pz["tem_refri"] == 1]
share = taxa_ponderada(pr, "tem_guarana")
print(f"Attach de refrigerante na pizza: {attach:.1%}")
print(f"Share de guaraná nas ocasiões pizza + refri: {share:.1%}")

Attach de refrigerante na pizza: 33.5%
Share de guaraná nas ocasiões pizza + refri: 29.4%


In [10]:
resumo = (
    pz.groupby("classe_renda", observed=True)
    .apply(lambda g: pd.Series({
        "ocasioes_pizza": len(g),
        "attach_refri": taxa_ponderada(g, "tem_refri"),
        "share_guarana": taxa_ponderada(g[g.tem_refri == 1], "tem_guarana") if g.tem_refri.sum() else np.nan,
    }), include_groups=False)
)
resumo.style.format({"attach_refri": "{:.1%}", "share_guarana": "{:.1%}", "ocasioes_pizza": "{:.0f}"})

,ocasioes_pizza,attach_refri,share_guarana
classe_renda,,,
E,136,48.9%,25.7%
D,186,44.4%,37.0%
C,348,34.6%,42.3%
B,382,39.5%,31.7%
A,658,25.4%,16.6%


8. Salvar a base

In [11]:
FEATURES_CONTEXTO = ["dia_semana", "dia_semana_num", "fim_de_semana", "hora", "periodo", "refeicao",
                     "local", "em_casa"]
FEATURES_PESSOA = ["idade", "sexo", "anos_estudo", "renda_pc", "classe_renda", "n_moradores",
                   "n_criancas", "n_adolescentes", "tem_crianca"]
FEATURES_TERRITORIO = ["uf", "regiao", "urbano"]
IDS = ["id_pessoa", "id_domicilio", "QUADRO", "estrato", "peso"]
TARGETS = ["y_pizza", "tem_refri", "tem_guarana", "pizza_refri", "pizza_guarana", "n_itens"]

final = base[IDS + FEATURES_CONTEXTO + FEATURES_PESSOA + FEATURES_TERRITORIO + TARGETS]
final.to_parquet(OUT / "base_ocasioes.parquet", index=False)
print("Salvo:", OUT / "base_ocasioes.parquet", final.shape)

Salvo: c:\Users\Inteli\Documents\caseAmbev\data\processed\base_ocasioes.parquet (363412, 31)
